In [36]:
import pandas as pd

In [ ]:
train_df = pd.read_csv("/Users/apple/Documents/Projects/Nasa RUL MLE/data/processed/train_clean.csv")
test_df = pd.read_csv("/Users/apple/Documents/Projects/Nasa RUL MLE/data/processed/test_clean.csv")
rul_df = pd.read_csv("/Users/apple/Documents/Projects/Nasa RUL MLE/data/processed/rul_clean.csv")

In [38]:
print("train_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)
print("rul_df shape:", rul_df.shape)

train_df shape: (20631, 21)
test_df shape: (13096, 20)
rul_df shape: (100, 1)


In [39]:
# Feature selection based on correlation
corr = train_df.corr()["rul"].abs()

selected_features = corr[corr > 0.2].index.tolist()
selected_features.remove("cycle")
selected_features.remove("rul")

selected_features

['sensor_2',
 'sensor_3',
 'sensor_4',
 'sensor_7',
 'sensor_8',
 'sensor_9',
 'sensor_11',
 'sensor_12',
 'sensor_13',
 'sensor_14',
 'sensor_15',
 'sensor_17',
 'sensor_20',
 'sensor_21']

In [40]:
train_df = train_df[["engine_id", "cycle"] + selected_features + ["rul"]]
test_df = test_df[["engine_id", "cycle"] + selected_features]

In [41]:
print("train_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)
print("rul_df shape:", rul_df.shape)

train_df shape: (20631, 17)
test_df shape: (13096, 16)
rul_df shape: (100, 1)


In [42]:
# Rolling features to capture degredation trend -> remove trend and smooth the signal
window = 5

for col in selected_features:
    train_df[f"{col}_rolling_mean"] = (
        train_df.groupby("engine_id")[col]
        .transform(lambda x: x.rolling(window, min_periods=1).mean().round(3))
    )

    test_df[f"{col}_rolling_mean"] = (
        test_df.groupby("engine_id")[col]
        .transform(lambda x: x.rolling(window, min_periods=1).mean().round(3))
    )

In [43]:
# Difference features -> capture change between cycles
for col in selected_features:
    train_df[f"{col}_diff"] = train_df.groupby("engine_id")[col].diff().fillna(0).round(3)
    test_df[f"{col}_diff"] = test_df.groupby("engine_id")[col].diff().fillna(0).round(3)

In [44]:
# Feature scaling -> put features on same scale -> fair learning across features
from sklearn.preprocessing import StandardScaler

features = [col for col in train_df.columns if col not in ["engine_id", "cycle", "rul"]]

scaler = StandardScaler()

train_df[features] = scaler.fit_transform(train_df[features])
test_df[features] = scaler.transform(test_df[features])

In [45]:
train_df

,engine_id,cycle,sensor_2,sensor_3,sensor_4,sensor_7,sensor_8,sensor_9,sensor_11,sensor_12,...,sensor_8_diff,sensor_9_diff,sensor_11_diff,sensor_12_diff,sensor_13_diff,sensor_14_diff,sensor_15_diff,sensor_17_diff,sensor_20_diff,sensor_21_diff
0,1,1,-1.721725,-0.134255,-0.925936,1.121141,-0.516338,-0.862813,-0.266467,0.334262,...,-0.021524,-0.035818,-0.028322,0.025498,-0.022331,-0.033494,-0.017790,-0.016332,0.018221,0.017621
1,1,2,-1.061780,0.211528,-0.643726,0.431930,-0.798093,-0.958818,-0.191583,1.174899,...,-0.489937,-0.395794,0.111298,1.489458,1.142862,-1.638525,0.403916,-0.016332,-0.402888,0.076423
2,1,3,-0.661813,-0.413166,-0.525953,1.008155,-0.234584,-0.557139,-1.015303,1.364721,...,0.915303,1.470306,-1.564147,0.356070,-0.954486,0.358197,-0.509781,-1.513867,-0.332704,-0.911441
3,1,4,-0.661813,-1.261314,-0.784831,1.222827,0.188048,-0.713826,-1.539489,1.961302,...,0.681096,-0.623326,-1.005665,1.064437,1.142862,0.101572,-1.774900,1.481203,-0.473073,0.370430
4,1,5,-0.621816,-1.251528,-0.301518,0.714393,-0.516338,-0.457059,-0.977861,1.052871,...,-1.192557,0.926946,1.018832,-1.556522,-0.954486,-0.040247,2.125885,0.732435,0.158590,0.370430
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20626,100,196,1.618000,1.216258,2.188375,-2.189329,1.315066,0.012547,1.980044,-2.607969,...,-0.958351,-0.743884,-1.075476,-0.493971,-0.488409,-1.226574,-1.985753,2.229970,2.474690,-2.557882
20627,100,197,1.717992,2.279706,2.738351,-2.833345,1.878576,-0.006020,1.867718,-2.350355,...,0.915303,-0.105436,-0.237753,0.474131,-0.954486,-0.281114,0.614770,-1.513867,-1.315291,2.205035
20628,100,198,1.478011,1.946971,2.138377,-2.742957,2.019453,0.029755,2.054927,-1.902919,...,0.212683,0.098323,0.320729,0.804703,0.443746,0.990755,1.774462,2.229970,1.000808,-2.640204
20629,100,199,1.098043,2.403666,1.955051,-3.036719,2.160330,0.383884,3.178182,-2.363913,...,0.212683,1.292016,2.065985,-0.777318,-0.255370,-0.429686,-0.931487,-2.262634,-1.034552,1.558219


In [46]:
test_df

,engine_id,cycle,sensor_2,sensor_3,sensor_4,sensor_7,sensor_8,sensor_9,sensor_11,sensor_12,...,sensor_8_diff,sensor_9_diff,sensor_11_diff,sensor_12_diff,sensor_13_diff,sensor_14_diff,sensor_15_diff,sensor_17_diff,sensor_20_diff,sensor_21_diff
0,1,1,0.678077,-0.853550,-1.191480,0.601408,-0.798093,-0.682579,-1.277396,0.415614,...,-0.021524,-0.035818,-0.028322,0.025498,-0.022331,-0.033494,-0.017790,-0.016332,0.018221,0.017621
1,1,2,-1.941707,-0.338137,-1.501467,1.674769,-1.220725,-0.490117,-0.154141,1.012195,...,-0.724144,0.685831,2.065985,1.064437,0.676785,3.133798,-0.896345,0.732435,1.141178,0.229307
2,1,3,-0.441831,-0.584426,-0.843717,0.838677,-0.657216,-0.375093,-0.154141,0.754581,...,0.915303,0.395473,-0.028322,-0.423134,-0.721447,-2.176536,2.231311,-0.016332,0.439330,0.311629
3,1,4,-0.481827,-1.044384,-0.279297,0.793483,-0.938970,-0.903570,-0.977861,-0.045381,...,-0.489937,-2.017382,-1.564147,-1.367624,0.443746,0.596813,-1.845185,-1.513867,-0.543258,-0.488071
4,1,5,-0.341839,-0.543650,-0.779276,0.895170,-1.220725,-0.937081,-0.865536,0.998637,...,-0.489937,-0.161470,0.181109,1.843641,-0.488409,-0.789862,0.368774,-0.765099,-0.051964,0.476273
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13091,100,194,1.118041,1.456023,0.761769,0.047780,-1.079848,3.492703,0.557254,-0.980929,...,-1.426764,0.386983,0.739591,-1.580134,-1.187525,-0.098775,1.001334,-0.765099,0.158590,-1.134887
13092,100,195,1.078044,0.842747,1.457295,-0.166892,-0.657216,3.416171,0.220277,-0.492817,...,0.681096,-0.322780,-0.656614,0.875539,2.075017,-0.580510,-0.720634,0.732435,-0.543258,0.958444
13093,100,196,1.518008,0.428459,-0.234855,-0.370266,0.188048,3.693768,0.107952,-0.316554,...,1.383717,1.005054,-0.237753,0.332458,-1.187525,1.404956,0.193063,-0.016332,0.369145,-0.829119
13094,100,197,1.158038,0.728573,1.158419,0.002586,-0.375461,3.786150,0.257719,-0.113174,...,-0.958351,0.310573,0.250919,0.379682,0.909823,0.695861,0.474201,-0.016332,0.298960,0.782040


In [47]:
print("train_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)

train_df shape: (20631, 45)
test_df shape: (13096, 44)


In [48]:
# save processed data
train_df.to_csv("/Users/apple/Documents/Projects/Nasa RUL MLE/data/processed/feature_engineered_train.csv", index=False)
test_df.to_csv("/Users/apple/Documents/Projects/Nasa RUL MLE/data/processed/feature_engineered_test.csv", index=False)